# Phase 2 — TrustFin-SLM Data Integrity, Benchmark Isolation, and Unified Data Contract
## Research: *Trustworthy Small Language Models for Financial Tasks*

**Purpose.** This notebook converts the existing exploratory work into a reproducible, research-safe data layer for Phase-3 (SLM baselines / QLoRA) and Phase-4 (TrustFin-SLM: verification, calibration, selective prediction, FinTrust evaluation).

### Phase-2 design rules

1. **FinQA**: use the canonical train/dev/test split; preserve the original question, full financial evidence, table, answer, executable result, gold program and gold evidence. Training examples are generated *directly* from official records — no hand-edited answer/program pairs.
2. **FiQA sentiment**: use the FiQA Task-1 style aspect-aware sentiment dataset (`TheFinAI/fiqa-sentiment-classification`) as the sanctioned alternative to FinNLI: train 822, dev (HF `valid`) 117, test 234, scores bounded in [-1, 1].
3. **FinTrust**: preserve the seven official trust dimensions and keep FinTrust **evaluation-only**. Do not coalesce all heterogeneous FinTrust files into a FinQA-shaped SFT table.
4. **Benchmark isolation**: FinQA test, FiQA test and every FinTrust evaluation item are excluded from SFT data and threshold selection.
5. **Stable interface**: downstream phases read JSONL/registry artifacts rather than reimplementing dataset loading.

### Primary references
- FinQA: Chen et al., EMNLP 2021 — https://aclanthology.org/2021.emnlp-main.300/
- FiQA sentiment dataset — https://huggingface.co/datasets/TheFinAI/fiqa-sentiment-classification
- FinTrust repository — https://github.com/HughieHu/FinTrust
- FinTrust dataset — https://huggingface.co/datasets/HughieHu/FinTrust

> **Note on benchmark selection**
> - The original FinNLI benchmark (Magomere et al., Findings of NAACL 2025) is not publicly downloadable (access by author request), so this project uses the FiQA sentiment benchmark as the sanctioned alternative — the same source frozen in the Phase-2.1 contract.
> - The earlier `FinNLI_EDA_Trustworthy_SLM.ipynb` loaded `TheFinAI/finben-fomc`, an FOMC stance dataset; FinBen-FOMC remains evaluation-only in Phase-2.1 and is never substituted here.
> - The previous FinQA 8-row curated sample contains answer/program mismatches and is therefore not used here.
> - The previous FinTrust generic flattening is replaced by a dimension/file registry that preserves task-specific schemas and official evaluators.


## 0. Environment and reproducibility

Run the installation cell once in a clean Python environment. Versions are intentionally broad enough for current systems while requiring modern `datasets` / `huggingface_hub` behavior.

In [1]:
# Uncomment if needed in a clean environment.
# %pip install -q "pandas>=2.0" "numpy>=1.24" "requests>=2.31" \
#     "datasets>=3.0" "huggingface_hub>=0.25" "pyarrow>=15" \
#     "matplotlib>=3.8" "scikit-learn>=1.4" "tqdm>=4.66"

In [2]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("pandas:", pd.__version__)

Python: 3.12.3
Platform: Linux-7.0.11-76070011-generic-x86_64-with-glibc2.39
pandas: 3.0.5


## 1. Configuration

### FiQA acquisition policy
Phase-2 trains on FinQA + FiQA sentiment. FiQA loads from the public Hugging Face repository `TheFinAI/fiqa-sentiment-classification` (override with the `FIQA_HF_REPO` environment variable if needed):

- HF splits `train` / `valid` / `test` map to Phase-2 logical splits `train` / `dev` / `test`.
- Expected counts: train 822, dev 117, test 234.

The unavailable FinNLI benchmark (author-request access only) is intentionally **not** substituted with FinBen-FOMC or any other dataset.


In [3]:
PROJECT_ROOT = Path(os.environ.get("TRUSTFIN_ROOT", "trustfin_slm_workspace")).resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "phase2"
EXPORT_DIR = ARTIFACT_DIR / "datasets"
EVAL_DIR = ARTIFACT_DIR / "eval"
MANIFEST_DIR = ARTIFACT_DIR / "manifests"
MODULE_DIR = ARTIFACT_DIR / "python"
AUDIT_DIR = ARTIFACT_DIR / "audits"

for p in [RAW_DIR, EXPORT_DIR, EVAL_DIR, MANIFEST_DIR, MODULE_DIR, AUDIT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# FinQA canonical GitHub release.
FINQA_BASE_URL = "https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset"
FINQA_EVALUATOR_URL = "https://raw.githubusercontent.com/czyssrs/FinQA/main/code/evaluate/evaluate.py"
FINQA_EXPECTED = {"train": 6251, "dev": 883, "test": 1147}

# FiQA Task-1 style aspect-aware sentiment (the sanctioned FinNLI alternative).
FIQA_HF_REPO = os.environ.get("FIQA_HF_REPO", "").strip() or "TheFinAI/fiqa-sentiment-classification"
FIQA_SPLIT_MAP = {"train": "train", "dev": "valid", "test": "test"}
FIQA_EXPECTED = {"train": 822, "dev": 117, "test": 234}
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()

# FinTrust is evaluation-only.
FINTRUST_HF_REPO = "HughieHu/FinTrust"
FINTRUST_GITHUB = "https://github.com/HughieHu/FinTrust.git"
FINTRUST_DIMENSIONS = [
    "trustfulness", "robustness", "safety", "fairness", "privacy",
    "transparency", "knowledge_discovery"
]

# Integrity settings.
DOWNLOAD_TIMEOUT = 90
FAIL_ON_CROSS_SPLIT_EXACT_DUPLICATE = True
RUN_OPTIONAL_NEAR_DUP_AUDIT = False   # set True for the final paper run if datasketch is installed
NEAR_DUP_THRESHOLD = 0.90

SCHEMA_VERSION = "trustfin-phase2-v1.1"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("FIQA_HF_REPO:", FIQA_HF_REPO)
print("FIQA_SPLIT_MAP:", FIQA_SPLIT_MAP)


PROJECT_ROOT: /home/heet18/Downloads/Research Papers
FIQA_HF_REPO: TheFinAI/fiqa-sentiment-classification
FIQA_SPLIT_MAP: {'train': 'train', 'dev': 'valid', 'test': 'test'}


## 2. Shared utility functions

These functions define deterministic serialization, hashing, schema normalization, file loading and integrity-report primitives. Phase-3/4 will reuse the exported helper module generated near the end of this notebook.

In [4]:
def json_default(x: Any):
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.floating,)): return float(x)
    if isinstance(x, np.ndarray): return x.tolist()
    if isinstance(x, Path): return str(x)
    if pd.isna(x) if not isinstance(x, (list, dict, tuple, set)) else False: return None
    return str(x)


def canonical_json(obj: Any) -> str:
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), default=json_default)


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def normalized_text(text: Any) -> str:
    s = "" if text is None else str(text)
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s


def input_signature(*parts: Any) -> str:
    return sha256_text("\n<SEP>\n".join(normalized_text(x) for x in parts))


def write_jsonl(records: Iterable[Mapping[str, Any]], path: Path) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    n = 0
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(dict(r), ensure_ascii=False, default=json_default) + "\n")
            n += 1
    return n


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    out = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                out.append(json.loads(line))
    return out


def save_json(obj: Any, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=json_default), encoding="utf-8")


def load_json_or_jsonl(path: Path) -> List[Dict[str, Any]]:
    text = path.read_text(encoding="utf-8-sig").strip()
    if not text:
        return []
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return [x for x in obj if isinstance(x, dict)]
        if isinstance(obj, dict):
            # Common wrappers.
            for key in ["data", "records", "examples", "items", "instances"]:
                if isinstance(obj.get(key), list):
                    return [x for x in obj[key] if isinstance(x, dict)]
            return [obj]
    except json.JSONDecodeError:
        pass
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return [x for x in rows if isinstance(x, dict)]


def load_tabular_file(path: Path) -> pd.DataFrame:
    ext = path.suffix.lower()
    if ext in {".json", ".jsonl"}:
        return pd.DataFrame(load_json_or_jsonl(path))
    if ext == ".csv":
        return pd.read_csv(path)
    if ext == ".tsv":
        return pd.read_csv(path, sep="\t")
    if ext == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file: {path}")


@dataclass
class Gate:
    name: str
    passed: bool
    observed: Any = None
    expected: Any = None
    severity: str = "ERROR"
    note: str = ""

GATES: List[Gate] = []

def add_gate(name: str, passed: bool, observed=None, expected=None, severity="ERROR", note=""):
    g = Gate(name, bool(passed), observed, expected, severity, note)
    GATES.append(g)
    status = "PASS" if g.passed else ("WARN" if severity == "WARNING" else "FAIL")
    print(f"[{status}] {name}")
    if not g.passed and note:
        print("       ", note)
    return g

# Part A — FinQA
## 3. Download and preserve the canonical FinQA split files

The downloader stores raw JSON files unchanged and records their SHA-256 hashes. This makes later experiments traceable to exact source artifacts.

In [5]:
FINQA_RAW_DIR = RAW_DIR / "finqa"
FINQA_RAW_DIR.mkdir(parents=True, exist_ok=True)


def download_file(url: str, path: Path, overwrite: bool = False) -> Path:
    if path.exists() and not overwrite:
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    r = requests.get(url, timeout=DOWNLOAD_TIMEOUT)
    r.raise_for_status()
    path.write_bytes(r.content)
    return path

finqa_raw_paths = {}
for split, filename in {"train": "train.json", "dev": "dev.json", "test": "test.json"}.items():
    p = FINQA_RAW_DIR / filename
    download_file(f"{FINQA_BASE_URL}/{filename}", p)
    finqa_raw_paths[split] = p

finqa_source_manifest = {
    split: {"path": str(path), "sha256": sha256_file(path), "bytes": path.stat().st_size}
    for split, path in finqa_raw_paths.items()
}
pd.DataFrame(finqa_source_manifest).T

,path,sha256,bytes
train,/home/heet18/Downloads/Research Papers/data/ra...,49f237eb9779b569473b26b08048867d04635a7cc39ad6...,78216616
dev,/home/heet18/Downloads/Research Papers/data/ra...,a847fb7e0d61a3125a1e2909852df6b89f1ee64d2c5ff1...,10954658
test,/home/heet18/Downloads/Research Papers/data/ra...,831dbfb2e785dbc227f895ce3f24046433467aec67b09d...,14395143


## 4. Parse FinQA without hand-editing gold targets

The canonical record keeps both **full input evidence** (available at inference time) and **gold evidence** (annotation metadata only). Gold evidence is never inserted as a privileged test-time prompt.

In [6]:
def finqa_table_to_text(table: Any) -> str:
    if not isinstance(table, list):
        return ""
    return "\n".join(" | ".join(map(str, row)) for row in table if isinstance(row, list))


def finqa_gold_evidence_to_text(gold_inds: Any) -> str:
    if isinstance(gold_inds, dict):
        return "\n".join(str(v) for _, v in sorted(gold_inds.items()))
    if isinstance(gold_inds, list):
        return "\n".join(map(str, gold_inds))
    return ""


def flatten_finqa(raw_rows: Sequence[Dict[str, Any]], split: str) -> pd.DataFrame:
    rows = []
    for item in raw_rows:
        qa = item.get("qa", {}) if isinstance(item.get("qa"), dict) else {}
        steps = qa.get("steps", []) or []
        final_step_result = None
        if steps and isinstance(steps[-1], dict):
            final_step_result = steps[-1].get("res")

        pre = item.get("pre_text", []) or []
        post = item.get("post_text", []) or []
        table = item.get("table", []) or []
        gold_inds = qa.get("gold_inds", {}) or {}

        rows.append({
            "id": str(item.get("id")),
            "split": split,
            "pre_text": pre,
            "post_text": post,
            "table": table,
            "question": qa.get("question"),
            "answer": qa.get("answer"),
            "exe_ans": qa.get("exe_ans"),
            "program": qa.get("program"),
            "program_re": qa.get("program_re"),
            "steps": steps,
            "gold_inds": gold_inds,
            "final_step_result": final_step_result,
            "context_text": "\n".join(map(str, pre + post)).strip(),
            "table_text": finqa_table_to_text(table),
            "gold_evidence_text": finqa_gold_evidence_to_text(gold_inds),
        })
    return pd.DataFrame(rows)

finqa_splits = {}
for split, path in finqa_raw_paths.items():
    raw = json.loads(path.read_text(encoding="utf-8"))
    finqa_splits[split] = flatten_finqa(raw, split)

finqa_df = pd.concat(finqa_splits.values(), ignore_index=True)
print(finqa_df.groupby("split").size())
display(finqa_df.head(2))

split
dev       883
test     1147
train    6251
dtype: int64


,id,split,pre_text,post_text,table,question,answer,exe_ans,program,program_re,steps,gold_inds,final_step_result,context_text,table_text,gold_evidence_text
0,ADI/2009/page_49.pdf-1,train,[interest rate to a variable interest rate bas...,[fair value of forward exchange contracts afte...,"[[, october 31 2009, november 1 2008], [fair v...",what is the the interest expense in 2009?,380,3.8,"divide(100, 100), divide(3.8, #0)","divide(3.8, divide(100, 100))","[{'op': 'divide1-1', 'arg1': '100', 'arg2': '1...",{'text_1': 'if libor changes by 100 basis poin...,380,interest rate to a variable interest rate base...,| october 31 2009 | november 1 2008\nfair val...,"if libor changes by 100 basis points , our ann..."
1,ABMD/2012/page_75.pdf-1,train,"[abiomed , inc ., and subsidiaries notes to co...",[the remaining unrecognized compensation expen...,"[[, number of shares ( in thousands ), weighte...","during the 2012 year , did the equity awards i...",,yes,"multiply(607, 18.13), multiply(#0, const_1000)...","greater(multiply(multiply(607, 18.13), const_1...","[{'op': 'multiply1-1', 'arg1': '607', 'arg2': ...",{'table_2': 'the granted of number of shares (...,yes,"abiomed , inc .\nand subsidiaries notes to con...",| number of shares ( in thousands ) | weighte...,the granted of number of shares ( in thousands...


## 5. FinQA integrity gates

We validate source counts, ID separation, required gold fields, final-step/result consistency, and exact input leakage. Duplicate *question strings* alone are reported but not treated as leakage because the associated report/table can differ; the leakage signature uses the complete question + context + table.

The **official FinQA evaluator** (`evaluate.py` from the canonical repo) executes every gold program; execution must equal `qa.exe_ans` (blocking). The raw final-step annotation audit is informational (`WARNING`): the final step may correspond to `qa.answer` or `qa.exe_ans`, and both representations are preserved.


In [7]:
def parse_number_loose(x: Any) -> Optional[float]:
    if x is None: return None
    s = str(x).strip().lower().replace(",", "").replace("$", "")
    if not s or s in {"none", "nan", "n/a"}: return None
    is_percent = s.endswith("%")
    s = s[:-1].strip() if is_percent else s
    try:
        v = float(s)
        return v / 100.0 if is_percent else v
    except ValueError:
        return None


def loose_equivalent(a: Any, b: Any, atol=1e-4, rtol=1e-3) -> bool:
    sa, sb = str(a).strip().lower(), str(b).strip().lower()
    if sa == sb: return True
    na, nb = parse_number_loose(a), parse_number_loose(b)
    if na is None or nb is None: return False
    # FinQA may represent percentages as either 23.6% or 0.236; parse handles that.
    return bool(np.isclose(na, nb, atol=atol, rtol=rtol))

# Source counts.
for split, expected in FINQA_EXPECTED.items():
    observed = len(finqa_splits[split])
    add_gate(f"FinQA {split} count", observed == expected, observed, expected)

# IDs.
add_gate("FinQA IDs globally unique", finqa_df["id"].nunique() == len(finqa_df),
         int(finqa_df["id"].nunique()), len(finqa_df))

required_cols = ["id", "question", "exe_ans", "program", "gold_inds", "table"]
for col in required_cols:
    missing = int(finqa_df[col].isna().sum()) if col in finqa_df else len(finqa_df)
    add_gate(f"FinQA required field present: {col}", missing == 0, missing, 0)

# The final step uses mixed official representations. Accept either the annotated
# human-facing answer or the executable numeric answer after normalization.
step_check = finqa_df[finqa_df["final_step_result"].notna()].copy()
step_check["final_matches_answer"] = step_check.apply(
    lambda r: loose_equivalent(r["final_step_result"], r["answer"]), axis=1
)
step_check["final_matches_exe_ans"] = step_check.apply(
    lambda r: loose_equivalent(r["final_step_result"], r["exe_ans"]), axis=1
)
step_check["final_consistent"] = step_check["final_matches_answer"] | step_check["final_matches_exe_ans"]
step_match_rate = float(step_check["final_consistent"].mean()) if len(step_check) else 0.0
answer_match_rate = float(step_check["final_matches_answer"].mean()) if len(step_check) else 0.0
exe_match_rate = float(step_check["final_matches_exe_ans"].mean()) if len(step_check) else 0.0
add_gate("FinQA final step matches an official answer representation", step_match_rate >= 0.99,
         round(step_match_rate, 5), ">=0.99",
         severity="WARNING",
         note="This source-annotation audit is non-blocking; the final step may correspond to qa.answer or qa.exe_ans, which are both preserved.")
print("Final-step/annotated-answer agreement:", round(answer_match_rate, 5))
print("Final-step/exe_ans agreement:", round(exe_match_rate, 5))

# Program vocabulary sanity check.
allowed_ops = {"add", "subtract", "multiply", "divide", "greater", "exp",
               "table_sum", "table_average", "table_max", "table_min"}
observed_ops = set()
for p in finqa_df["program"].dropna().astype(str):
    observed_ops.update(re.findall(r"([A-Za-z_][A-Za-z0-9_]*)\(", p))
unexpected_ops = sorted(observed_ops - allowed_ops)
add_gate("FinQA program operation vocabulary recognized", len(unexpected_ops) == 0,
         unexpected_ops, [])

# Exact full-input signatures across splits.
finqa_df["input_signature"] = finqa_df.apply(
    lambda r: input_signature(r["question"], r["context_text"], r["table_text"]), axis=1
)
sig_splits = finqa_df.groupby("input_signature")["split"].nunique()
cross_split_sig_count = int((sig_splits > 1).sum())
add_gate("FinQA no exact full-input leakage across splits", cross_split_sig_count == 0,
         cross_split_sig_count, 0)

print("Duplicate question strings (informational):", int(finqa_df["question"].duplicated().sum()))
print("Final-step/official-answer mismatches to inspect:", int((~step_check["final_consistent"]).sum()))
display(step_check.loc[~step_check["final_consistent"],
                       ["id", "split", "question", "answer", "exe_ans", "final_step_result", "program"]].head(20))

# ---- Official FinQA evaluator: blocking execution-correctness gates ----
import importlib.util

FINQA_EVALUATOR_PATH = RAW_DIR / "finqa_official_evaluator" / "evaluate.py"
download_file(FINQA_EVALUATOR_URL, FINQA_EVALUATOR_PATH)

spec = importlib.util.spec_from_file_location("finqa_official_evaluate", FINQA_EVALUATOR_PATH)
finqa_official = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(finqa_official)
assert hasattr(finqa_official, "program_tokenization")
assert hasattr(finqa_official, "eval_program")


def validate_finqa_gold_execution(raw_path: Path, split: str) -> pd.DataFrame:
    raw = json.loads(raw_path.read_text(encoding="utf-8"))
    rows = []
    for item in tqdm(raw, desc=f"Official FinQA executor: {split}"):
        qa = item["qa"]
        tokens = finqa_official.program_tokenization(qa["program"])
        invalid_flag, executed = finqa_official.eval_program(tokens, item["table"])
        expected = qa["exe_ans"]
        rows.append({
            "id": str(item["id"]),
            "split": split,
            "invalid_flag": int(invalid_flag),
            "executed": executed,
            "expected_exe_ans": expected,
            # Use the official evaluator's equality rule: exact Python equality.
            "official_match": bool(invalid_flag == 0 and executed == expected),
            "program": qa["program"],
        })
    return pd.DataFrame(rows)


finqa_official_validation = {}
for split, path in finqa_raw_paths.items():
    df_exec = validate_finqa_gold_execution(path, split)
    finqa_official_validation[split] = df_exec

    invalid_n = int((df_exec["invalid_flag"] != 0).sum())
    mismatch_n = int((~df_exec["official_match"]).sum())
    add_gate(f"FinQA {split}: official gold programs execute successfully", invalid_n == 0, invalid_n, 0)
    add_gate(f"FinQA {split}: official execution equals qa.exe_ans", mismatch_n == 0, mismatch_n, 0)

finqa_exec_audit_df = pd.concat(finqa_official_validation.values(), ignore_index=True)
finqa_exec_audit_path = AUDIT_DIR / "finqa_official_gold_execution_audit.csv"
finqa_exec_audit_df.to_csv(finqa_exec_audit_path, index=False)

print("Official execution audit:", finqa_exec_audit_path)
display(finqa_exec_audit_df.groupby("split").agg(
    n=("id", "size"),
    invalid=("invalid_flag", lambda s: int((s != 0).sum())),
    official_match=("official_match", "sum"),
))

[PASS] FinQA train count
[PASS] FinQA dev count
[PASS] FinQA test count
[PASS] FinQA IDs globally unique
[PASS] FinQA required field present: id
[PASS] FinQA required field present: question
[PASS] FinQA required field present: exe_ans
[PASS] FinQA required field present: program
[PASS] FinQA required field present: gold_inds
[PASS] FinQA required field present: table
[WARN] FinQA final step matches an official answer representation
        This source-annotation audit is non-blocking; the final step may correspond to qa.answer or qa.exe_ans, which are both preserved.
Final-step/annotated-answer agreement: 0.93817
Final-step/exe_ans agreement: 0.6469
[PASS] FinQA program operation vocabulary recognized


[PASS] FinQA no exact full-input leakage across splits
Duplicate question strings (informational): 163
Final-step/official-answer mismatches to inspect: 144


,id,split,question,answer,exe_ans,final_step_result,program
36,RE/2012/page_31.pdf-4,train,"as of december 31 , 2012 what was the percenta...",-11.5%,-0.11349,-11.3%,"subtract(425.7, 480.2), divide(#0, 480.2)"
77,HII/2013/page_116.pdf-1,train,"for the year ended december 31 2013 , what was...",6.9%,0.06892,6.9,"divide(116, 1683)"
129,SNA/2013/page_33.pdf-1,train,how is the cash flow statement from financing ...,531600,531600.0,-531600,"multiply(5000, 106.32)"
219,VTR/2006/page_88.pdf-4,train,what was the percent of growth in maturities f...,-40.1%,-0.40167,40.1%,"subtract(265915, 372725), divide(#0, 265915)"
229,CNC/2005/page_39.pdf-1,train,what was the percentage change in the allowanc...,-24%,-0.23888,24%,"subtract(462, 607), divide(#0, 607)"
314,BLK/2014/page_33.pdf-2,train,what percent change did the fx impact have on ...,-3.4%,-0.03656,.034,"subtract(377837, 13326), divide(377837, #0), s..."
326,AWK/2014/page_132.pdf-2,train,what was the ratio of the increase in the serv...,,0.05608,0.06,"divide(5943, 105967)"
331,JPM/2014/page_128.pdf-1,train,what percentage of the total exposure net of a...,23.5%,0.23479,23.5,"divide(13940, 59371)"
371,RSG/2014/page_22.pdf-1,train,what is the percent of the residential vehicle...,48.1%,0.48101,15.1%,"divide(7600, 15800)"
424,SNA/2012/page_93.pdf-2,train,what was the average undistributed earnings of...,431.7\\n,649.05,431.7,"add(492.2, 416.4), add(#0, 386.5), add(#1, con..."


Official FinQA executor: train:   0%|          | 0/6251 [00:00<?, ?it/s]

[PASS] FinQA train: official gold programs execute successfully
[PASS] FinQA train: official execution equals qa.exe_ans


Official FinQA executor: dev:   0%|          | 0/883 [00:00<?, ?it/s]

[PASS] FinQA dev: official gold programs execute successfully
[PASS] FinQA dev: official execution equals qa.exe_ans


Official FinQA executor: test:   0%|          | 0/1147 [00:00<?, ?it/s]

[PASS] FinQA test: official gold programs execute successfully
[PASS] FinQA test: official execution equals qa.exe_ans
Official execution audit: /home/heet18/Downloads/Research Papers/artifacts/phase2/audits/finqa_official_gold_execution_audit.csv


,n,invalid,official_match
split,,,
dev,883,0,883
test,1147,0,1147
train,6251,0,6251


## 6. FinQA canonical Phase-3/4 records

The model input uses **full report text + table**, never gold evidence. For program-faithful supervision, `ANSWER` uses FinQA's official `qa.exe_ans` execution result; the separate human-readable `qa.answer` is retained in metadata because the two are not guaranteed to be numerically identical. The target is structured and machine-checkable:

```text
PROGRAM: <gold program>
ANSWER: <gold answer>
```

This directly supports Phase-4 execution verification without requiring natural-language chain-of-thought supervision.

In [8]:
FINQA_SYSTEM = (
    "You are a financial reasoning assistant. Use only the supplied financial evidence. "
    "Return a valid FinQA program and the final answer."
)


def make_finqa_record(row: pd.Series) -> Dict[str, Any]:
    input_text = (
        f"Financial text:\n{row['context_text']}\n\n"
        f"Financial table:\n{row['table_text']}\n\n"
        f"Question:\n{row['question']}"
    ).strip()
    target = f"PROGRAM: {row['program']}\nANSWER: {row['exe_ans']}"
    return {
        "record_id": f"finqa::{row['id']}",
        "source_id": row["id"],
        "dataset": "FinQA",
        "task": "financial_numerical_reasoning",
        "split": row["split"],
        "system": FINQA_SYSTEM,
        "instruction": "Solve the financial question from the supplied evidence.",
        "input": input_text,
        "target": target,
        "target_answer": row["exe_ans"],
        "target_label": None,
        "gold_program": row["program"],
        "gold_evidence": row["gold_inds"],
        "metadata": {
            "schema_version": SCHEMA_VERSION,
            "has_table": bool(row["table"]),
            "annotated_answer": row["answer"],
            "execution_answer": row["exe_ans"],
            "input_signature": row["input_signature"],
            "evaluation_only": row["split"] == "test",
        },
    }

finqa_records = {
    split: [make_finqa_record(r) for _, r in finqa_splits[split].assign(
        input_signature=finqa_splits[split].apply(
            lambda x: input_signature(x["question"], x["context_text"], x["table_text"]), axis=1
        )
    ).iterrows()]
    for split in ["train", "dev", "test"]
}

print({k: len(v) for k, v in finqa_records.items()})
print(json.dumps(finqa_records["train"][0], indent=2, ensure_ascii=False)[:2000])

{'train': 6251, 'dev': 883, 'test': 1147}
{
  "record_id": "finqa::ADI/2009/page_49.pdf-1",
  "source_id": "ADI/2009/page_49.pdf-1",
  "dataset": "FinQA",
  "task": "financial_numerical_reasoning",
  "split": "train",
  "system": "You are a financial reasoning assistant. Use only the supplied financial evidence. Return a valid FinQA program and the final answer.",
  "instruction": "Solve the financial question from the supplied evidence.",
  "input": "Financial text:\ninterest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) .\nif libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million .\nforeign currency exposure as more fully described in note 2i .\nin the notes to consolidated financial statements contained in item 8 of this annual report on form 10-k , we regularly hedge our non-u.s .\ndollar-based exposures by entering into forward foreign currency exchange contr

# Part B — FiQA Sentiment (FinNLI alternative)
## 7. FiQA source discovery

FiQA loads from `TheFinAI/fiqa-sentiment-classification` through the same HuggingFace loading path Phase-2.1 already validates. HF split names are mapped to Phase-2 logical splits (`valid` → `dev`), and the required schema columns are checked before normalization.


In [9]:
ALLOWED_TABULAR_EXT = {".csv", ".tsv", ".json", ".jsonl", ".parquet"}


def hf_dataset_to_pandas(repo_id: str, hf_split: str) -> pd.DataFrame:
    from datasets import load_dataset
    ds = load_dataset(repo_id, split=hf_split, token=HF_TOKEN or None)
    return ds.to_pandas()


fiqa_raw: Dict[str, pd.DataFrame] = {}
FIQA_AVAILABLE = False
try:
    for logical_split, hf_split in FIQA_SPLIT_MAP.items():
        fiqa_raw[logical_split] = hf_dataset_to_pandas(FIQA_HF_REPO, hf_split)
    FIQA_AVAILABLE = True
except Exception as e:
    print("FiQA source load failed:", repr(e))

if FIQA_AVAILABLE:
    print({k: len(v) for k, v in fiqa_raw.items()})
    for k, v in fiqa_raw.items():
        print(k, "columns:", list(v.columns))
else:
    print("FiQA files were not loaded.")
    print("Check network/HF cache or set FIQA_HF_REPO to an explicit repository ID.")

fiqa_source_manifest = {
    split: {
        "mode": "huggingface",
        "hf_repo": FIQA_HF_REPO,
        "hf_split": FIQA_SPLIT_MAP[split],
        "rows": int(len(df)),
        "sha256": sha256_text(
            canonical_json(
                sorted(
                    df.to_dict("records"),
                    key=lambda r: str(r.get("_id", r.get("id", ""))),
                )
            )
        ),
    }
    for split, df in fiqa_raw.items()
} if FIQA_AVAILABLE else {}


{'train': 822, 'dev': 117, 'test': 234}
train columns: ['_id', 'sentence', 'target', 'aspect', 'score', 'type']
dev columns: ['_id', 'sentence', 'target', 'aspect', 'score', 'type']
test columns: ['_id', 'sentence', 'target', 'aspect', 'score', 'type']


## 8. FiQA schema normalization and integrity gates

Required columns: `_id`, `sentence`, `target`, `aspect`, `score`, `type`. Within-split duplicate `_id` values receive deterministic `_0`, `_1`, … suffixes. Scores must be numeric and bounded in [-1, 1]; no label vocabulary is guessed.


In [10]:
FIQA_REQUIRED_COLUMNS = ["_id", "sentence", "target", "aspect", "score", "type"]


def normalize_fiqa_frame(df: pd.DataFrame, split: str) -> pd.DataFrame:
    missing = [c for c in FIQA_REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"FiQA {split}: missing required columns {missing}. Available={list(df.columns)}")

    out = pd.DataFrame()
    raw_ids = df["_id"].astype(str).tolist()
    id_counts = Counter(raw_ids)
    id_seen: Dict[str, int] = {}
    unique_ids = []
    for rid in raw_ids:
        if id_counts[rid] > 1:
            idx = id_seen.get(rid, 0)
            id_seen[rid] = idx + 1
            unique_ids.append(f"{rid}_{idx}")
        else:
            unique_ids.append(rid)
    out["id"] = unique_ids
    out["split"] = split
    out["sentence"] = df["sentence"].astype(str).str.strip()
    out["entity_target"] = df["target"].astype(str).str.strip()
    out["aspect"] = df["aspect"].astype(str).str.strip()
    out["score"] = pd.to_numeric(df["score"], errors="coerce")
    out["text_type"] = df["type"].astype(str).str.strip().str.lower()
    out["input_signature"] = out.apply(
        lambda r: input_signature(r["sentence"], r["entity_target"], r["aspect"]), axis=1
    )
    return out


fiqa_splits: Dict[str, pd.DataFrame] = {}
if FIQA_AVAILABLE:
    for split in ["train", "dev", "test"]:
        if split not in fiqa_raw:
            raise ValueError(f"FiQA is missing required split: {split}")
        fiqa_splits[split] = normalize_fiqa_frame(fiqa_raw[split], split)

    fiqa_df = pd.concat(list(fiqa_splits.values()), ignore_index=True)
    display(fiqa_df.head())

    for split, expected in FIQA_EXPECTED.items():
        observed = len(fiqa_splits[split])
        add_gate(f"FiQA {split} count", observed == expected, observed, expected)

    add_gate(
        "FiQA IDs globally unique",
        fiqa_df["id"].nunique() == len(fiqa_df),
        int(fiqa_df["id"].nunique()),
        len(fiqa_df),
        severity="WARNING",
        note="If upstream IDs are reused across splits, record IDs still include the split to remain collision-safe.",
    )
    add_gate("FiQA sentence non-empty", bool(fiqa_df["sentence"].str.len().gt(0).all()))
    add_gate("FiQA target entity non-empty", bool(fiqa_df["entity_target"].str.len().gt(0).all()))
    add_gate("FiQA aspect non-empty", bool(fiqa_df["aspect"].str.len().gt(0).all()))
    add_gate("FiQA score numeric", int(fiqa_df["score"].isna().sum()) == 0, int(fiqa_df["score"].isna().sum()), 0)
    add_gate(
        "FiQA score bounded in [-1, 1]",
        bool(fiqa_df["score"].between(-1.0, 1.0, inclusive="both").all()),
        [float(fiqa_df["score"].min()), float(fiqa_df["score"].max())],
        [-1.0, 1.0],
    )

    cross = fiqa_df.groupby("input_signature")["split"].nunique()
    fiqa_cross_exact = int((cross > 1).sum())
    add_gate(
        "FiQA no exact sentence-aspect leakage across splits",
        fiqa_cross_exact == 0,
        fiqa_cross_exact,
        0,
    )
else:
    add_gate(
        "FiQA source configured",
        False,
        observed="not found",
        expected=FIQA_HF_REPO,
        note="Phase-2 cannot pass until the FiQA sentiment source is reachable.",
    )
    fiqa_df = pd.DataFrame()


,id,split,sentence,entity_target,aspect,score,text_type,input_signature
0,1,train,Royal Mail chairman Donald Brydon set to step ...,Royal Mail,Corporate/Appointment,-0.374,headline,5926875038c97be56d492889ec9146040576f9e02490e8...
1,100,train,Slump in Weir leads FTSE down from record high,Weir,Market/Volatility/Volatility,-0.827,headline,efe3ed2925ef5765f7b1028e6614b565746a3626ef056e...
2,1000,train,AstraZeneca wins FDA approval for key new lung...,AstraZeneca,Corporate/Regulatory,0.549,headline,ee86c612d0e7cdfa3f80c5dd8124959bb14f2558ee3b51...
3,1002,train,UPDATE 1-Lloyds to cut 945 jobs as part of 3-y...,Lloyds,Corporate/Strategy,-0.266,headline,71c43d510664754420e4002b5cab3c989b518c73d8e1d7...
4,1005,train,Standard Chartered Shifts Emerging-Markets Str...,Standard Chartered,Corporate/Strategy,-0.461,headline,09cd8c455068607bf0fd043c8e98a3a3c826d977ff132e...


[PASS] FiQA train count
[PASS] FiQA dev count
[PASS] FiQA test count
[PASS] FiQA IDs globally unique
[PASS] FiQA sentence non-empty
[PASS] FiQA target entity non-empty
[PASS] FiQA aspect non-empty
[PASS] FiQA score numeric
[PASS] FiQA score bounded in [-1, 1]
[PASS] FiQA no exact sentence-aspect leakage across splits


## 9. FiQA research diagnostics

These checks are lightweight but useful for the paper: score distribution per split, sentence lengths, text-type balance and aspect coverage. They also catch obvious schema mistakes before training.


In [11]:
if FIQA_AVAILABLE:
    stats = []
    for split, df in fiqa_splits.items():
        tmp = df.copy()
        tmp["sentence_words"] = tmp["sentence"].str.split().str.len()
        stats.append({
            "split": split,
            "n": len(tmp),
            "sentence_words_mean": round(tmp["sentence_words"].mean(), 2),
            "sentence_words_p95": int(tmp["sentence_words"].quantile(.95)),
            "score_mean": round(float(tmp["score"].mean()), 4),
            "score_min": float(tmp["score"].min()),
            "score_max": float(tmp["score"].max()),
            "aspects": int(tmp["aspect"].nunique()),
            "entities": int(tmp["entity_target"].nunique()),
        })
    display(pd.DataFrame(stats))
    display(pd.crosstab(fiqa_df["split"], fiqa_df["text_type"]))
    display(fiqa_df["aspect"].value_counts().head(20).rename("count").to_frame())


,split,n,sentence_words_mean,sentence_words_p95,score_mean,score_min,score_max,aspects,entities
0,train,822,12.03,23,0.1423,-0.938,0.937,117,448
1,dev,117,11.54,21,0.0725,-0.626,0.721,43,66
2,test,234,11.51,21,0.0748,-0.812,0.975,67,114


text_type,headline,post
split,,
dev,50,67
test,99,135
train,349,473


,count
aspect,
Stock/Price Action/Bullish,88
Stock/Price Action,81
Stock/Technical Analysis,78
Stock/Price Action/Bullish/Bullish Behavior,72
Corporate/Sales,48
Stock/Price Action/Bearish,48
Corporate/M&A/M&A,45
Stock/Price Action/Bearish/Bearish Behavior,43
Stock/Price Action/Bullish/Bull Position,41


## 10. FiQA canonical Phase-3/4 records

FiQA uses score-only supervision (`SENTIMENT_SCORE: <number>`). No generated rationale is fabricated. This keeps the training target faithful to the benchmark annotation.


In [12]:
FIQA_SYSTEM = (
    "You are a financial sentiment analysis assistant. Estimate the sentiment toward "
    "the specified financial target for the specified aspect. Return a numeric score "
    "from -1 (most negative) to +1 (most positive)."
)


def format_fiqa_score(x: float) -> str:
    return format(float(x), ".10g")


def make_fiqa_record(row: pd.Series) -> Dict[str, Any]:
    inp = (
        f"Financial text:\n{row['sentence']}\n\n"
        f"Target entity:\n{row['entity_target']}\n\n"
        f"Aspect:\n{row['aspect']}"
    ).strip()
    score = float(row["score"])
    return {
        "record_id": f"fiqa::{row['split']}::{row['id']}",
        "source_id": row["id"],
        "dataset": "FiQA",
        "task": "financial_sentiment_regression",
        "split": row["split"],
        "system": FIQA_SYSTEM,
        "instruction": "Return only `SENTIMENT_SCORE: <number>` with a value from -1 to +1.",
        "input": inp,
        "target": f"SENTIMENT_SCORE: {format_fiqa_score(score)}",
        "target_answer": score,
        "target_label": None,
        "gold_program": None,
        "gold_evidence": None,
        "metadata": {
            "schema_version": SCHEMA_VERSION,
            "entity_target": row["entity_target"],
            "aspect": row["aspect"],
            "text_type": row["text_type"],
            "sentiment_score": score,
            "input_signature": row["input_signature"],
            "evaluation_only": row["split"] == "test",
        },
    }


fiqa_records = {"train": [], "dev": [], "test": []}
if FIQA_AVAILABLE:
    for split in ["train", "dev", "test"]:
        fiqa_records[split] = [make_fiqa_record(r) for _, r in fiqa_splits[split].iterrows()]
    print({k: len(v) for k, v in fiqa_records.items()})
    print(json.dumps(fiqa_records["train"][0], indent=2, ensure_ascii=False)[:2000])


{'train': 822, 'dev': 117, 'test': 234}
{
  "record_id": "fiqa::train::1",
  "source_id": "1",
  "dataset": "FiQA",
  "task": "financial_sentiment_regression",
  "split": "train",
  "system": "You are a financial sentiment analysis assistant. Estimate the sentiment toward the specified financial target for the specified aspect. Return a numeric score from -1 (most negative) to +1 (most positive).",
  "instruction": "Return only `SENTIMENT_SCORE: <number>` with a value from -1 to +1.",
  "input": "Financial text:\nRoyal Mail chairman Donald Brydon set to step down\n\nTarget entity:\nRoyal Mail\n\nAspect:\nCorporate/Appointment",
  "target": "SENTIMENT_SCORE: -0.374",
  "target_answer": -0.374,
  "target_label": null,
  "gold_program": null,
  "gold_evidence": null,
  "metadata": {
    "schema_version": "trustfin-phase2-v1.1",
    "entity_target": "Royal Mail",
    "aspect": "Corporate/Appointment",
    "text_type": "headline",
    "sentiment_score": -0.374,
    "input_signature": "59268

# Part C — FinTrust evaluation registry
## 11. Snapshot FinTrust without converting it into SFT data

The benchmark spans **trustfulness, robustness, safety, fairness, privacy, transparency, and knowledge discovery**. Its files are heterogeneous by design. Phase-2 therefore creates a manifest of files, dimensions, schemas, hashes and official evaluation scripts instead of inventing one common answer column.

In [13]:
FINTRUST_RAW_DIR = RAW_DIR / "fintrust_hf"
FINTRUST_EVAL_CODE_DIR = RAW_DIR / "fintrust_eval_code"


def normalize_dimension_name(name: str) -> str:
    return name.lower().replace("-", "_").replace(" ", "_")


def snapshot_fintrust_hf() -> Path:
    from huggingface_hub import snapshot_download
    path = snapshot_download(
        repo_id=FINTRUST_HF_REPO,
        repo_type="dataset",
        local_dir=str(FINTRUST_RAW_DIR),
    )
    return Path(path)


def clone_or_update_fintrust_eval_repo() -> Optional[Path]:
    try:
        if (FINTRUST_EVAL_CODE_DIR / ".git").exists():
            subprocess.run(["git", "-C", str(FINTRUST_EVAL_CODE_DIR), "pull", "--ff-only"],
                           check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        elif not FINTRUST_EVAL_CODE_DIR.exists() or not any(FINTRUST_EVAL_CODE_DIR.iterdir()):
            subprocess.run(["git", "clone", "--depth", "1", FINTRUST_GITHUB, str(FINTRUST_EVAL_CODE_DIR)],
                           check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        return FINTRUST_EVAL_CODE_DIR
    except Exception as e:
        print("FinTrust evaluator repo clone/update warning:", repr(e))
        return None

FINTRUST_AVAILABLE = True
try:
    fintrust_snapshot = snapshot_fintrust_hf()
except Exception as e:
    FINTRUST_AVAILABLE = False
    fintrust_snapshot = FINTRUST_RAW_DIR
    print("FinTrust Hugging Face snapshot failed:", repr(e))

fintrust_eval_code = clone_or_update_fintrust_eval_repo()
print("FinTrust data snapshot:", fintrust_snapshot)
print("FinTrust evaluator code:", fintrust_eval_code)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

FinTrust data snapshot: /home/heet18/Downloads/Research Papers/data/raw/fintrust_hf
FinTrust evaluator code: /home/heet18/Downloads/Research Papers/data/raw/fintrust_eval_code


## 12. Build dimension-aware FinTrust file manifest

Only manifest metadata is normalized. Raw records remain untouched. Phase-4 can select a dimension/file and then apply the corresponding official `postprocess_response.py` logic.

In [14]:
def infer_fintrust_dimension(path: Path, root: Path) -> Optional[str]:
    try:
        rel = path.relative_to(root)
    except ValueError:
        rel = path
    parts = [normalize_dimension_name(x) for x in rel.parts]
    joined = "_".join(parts)
    for dim in FINTRUST_DIMENSIONS:
        if dim in parts or dim in joined:
            return dim
    # tolerate hyphenated official directory naming
    if "knowledge_discovery" in joined:
        return "knowledge_discovery"
    return None


def classify_fintrust_file_role(path: Path) -> str:
    n = path.name.lower()
    if any(k in n for k in ["response", "output", "result", "prediction"]): return "generated_or_result"
    if any(k in n for k in ["prompt", "test", "sample", "dataset", "question", "data"]): return "candidate_eval_data"
    return "unknown"


def summarize_schema_from_records(records: List[Dict[str, Any]], limit: int = 200) -> List[str]:
    keys = set()
    for r in records[:limit]:
        keys.update(map(str, r.keys()))
    return sorted(keys)

fintrust_manifest_rows = []
if FINTRUST_AVAILABLE:
    data_files = [
        p for p in fintrust_snapshot.rglob("*")
        if p.is_file() and p.suffix.lower() in ALLOWED_TABULAR_EXT and ".cache" not in p.parts
    ]
    for p in tqdm(data_files, desc="FinTrust files"):
        dim = infer_fintrust_dimension(p, fintrust_snapshot)
        try:
            df = load_tabular_file(p)
            schema = sorted(map(str, df.columns))
            n_rows = len(df)
            load_error = None
        except Exception as e:
            schema, n_rows, load_error = [], None, repr(e)
        fintrust_manifest_rows.append({
            "dimension": dim,
            "relative_path": str(p.relative_to(fintrust_snapshot)),
            "absolute_path": str(p.resolve()),
            "role": classify_fintrust_file_role(p),
            "rows": n_rows,
            "columns": schema,
            "sha256": sha256_file(p),
            "load_error": load_error,
            "evaluation_only": True,
        })

fintrust_manifest_df = pd.DataFrame(fintrust_manifest_rows)
if len(fintrust_manifest_df):
    display(fintrust_manifest_df[["dimension", "relative_path", "role", "rows", "load_error"]])
    display(fintrust_manifest_df.groupby("dimension", dropna=False)["rows"].sum(min_count=1).rename("raw_rows"))

FinTrust files:   0%|          | 0/12 [00:00<?, ?it/s]

,dimension,relative_path,role,rows,load_error
0,privacy,privacy/privacy_evaluation_1000.json,unknown,1000,None
1,transparency,transparency/transparency_evaluation_647.json,unknown,647,None
2,robustness,robustness/robustness_evaluation_220.json,unknown,220,None
3,knowledge_discovery,knowledge-discovery/knowledge_discovery_evalua...,unknown,1000,None
4,fairness,fairness/personal-level_data/fairness_personal...,unknown,1000,None
5,fairness,fairness/company-level_data/fairness_company_l...,unknown,1000,None
6,safety,safety/deep-inception/safety_deep_inception_ev...,unknown,1000,None
7,safety,safety/language_barrier/safety_language_barrie...,unknown,1000,None
8,safety,safety/in-context attack/safety_in_context_att...,unknown,1,None
9,safety,safety/plain_question/safety_plain_question_ev...,candidate_eval_data,1,None


dimension
fairness               2000
knowledge_discovery    1000
privacy                1000
robustness              220
safety                 2002
transparency            647
trustfulness           1800
Name: raw_rows, dtype: int64

## 13. FinTrust integrity gates

The repository states that the benchmark covers seven dimensions and 15,680 answer pairs. Because file schemas and the meaning of an “answer pair” differ by task, **raw dataframe row totals are not forced to equal 15,680**. Instead we require all seven dimensions to be represented and we preserve the official per-dimension evaluation code whenever available.

In [15]:
if FINTRUST_AVAILABLE and len(fintrust_manifest_df):
    observed_dims = sorted(d for d in fintrust_manifest_df["dimension"].dropna().unique())
    add_gate("FinTrust all seven dimensions represented", set(observed_dims) == set(FINTRUST_DIMENSIONS),
             observed_dims, sorted(FINTRUST_DIMENSIONS))
    add_gate("FinTrust data files load without parser errors",
             int(fintrust_manifest_df["load_error"].notna().sum()) == 0,
             int(fintrust_manifest_df["load_error"].notna().sum()), 0)
else:
    add_gate("FinTrust dataset snapshot available", False, observed="unavailable", expected=FINTRUST_HF_REPO)

if fintrust_eval_code and fintrust_eval_code.exists():
    script_dims = []
    for dim in FINTRUST_DIMENSIONS:
        candidates = [dim, dim.replace("_", "-")]
        found = any((fintrust_eval_code / c / "postprocess_response.py").exists() for c in candidates)
        if found: script_dims.append(dim)
    add_gate("FinTrust official postprocess scripts found for seven dimensions",
             set(script_dims) == set(FINTRUST_DIMENSIONS), script_dims, FINTRUST_DIMENSIONS,
             severity="WARNING",
             note="A warning here does not justify replacing official task-specific evaluation with a generic metric.")
else:
    add_gate("FinTrust official evaluator repository available", False,
             observed="git clone unavailable", expected=FINTRUST_GITHUB, severity="WARNING")

[PASS] FinTrust all seven dimensions represented
[PASS] FinTrust data files load without parser errors
[WARN] FinTrust official postprocess scripts found for seven dimensions
        A warning here does not justify replacing official task-specific evaluation with a generic metric.


# Part D — Cross-dataset leakage and unified Phase-3/4 contract
## 14. Build protected train / validation / evaluation pools

**Training pool** = FinQA train + FiQA train.
**Validation pool** = FinQA dev + FiQA dev.
**Protected evaluation** = FinQA test + FiQA test + all FinTrust data.

FinTrust is never appended to the SFT pool.


In [16]:
train_records = list(finqa_records["train"]) + list(fiqa_records["train"])
validation_records = list(finqa_records["dev"]) + list(fiqa_records["dev"])
finqa_test_records = list(finqa_records["test"])
fiqa_test_records = list(fiqa_records["test"])

print("Train records:", len(train_records), Counter(r["dataset"] for r in train_records))
print("Validation records:", len(validation_records), Counter(r["dataset"] for r in validation_records))
print("FinQA test:", len(finqa_test_records))
print("FiQA test:", len(fiqa_test_records))

add_gate("No FinTrust records in training", all(r["dataset"] != "FinTrust" for r in train_records))
add_gate("No evaluation_only records in training", all(not r["metadata"].get("evaluation_only", False) for r in train_records))


Train records: 7073 Counter({'FinQA': 6251, 'FiQA': 822})
Validation records: 1000 Counter({'FinQA': 883, 'FiQA': 117})
FinQA test: 1147
FiQA test: 234
[PASS] No FinTrust records in training
[PASS] No evaluation_only records in training


Gate(name='No evaluation_only records in training', passed=True, observed=None, expected=None, severity='ERROR', note='')

## 15. Mandatory exact leakage audit

Leakage is checked using normalized **model inputs**, not source IDs alone. We test train↔validation, train↔FinQA-test and train↔FiQA-test intersections.


In [17]:
def record_input_sig(r: Mapping[str, Any]) -> str:
    return r.get("metadata", {}).get("input_signature") or input_signature(r.get("input", ""))


def sig_set(records: Sequence[Mapping[str, Any]]) -> set:
    return {record_input_sig(r) for r in records}

train_sigs = sig_set(train_records)
val_sigs = sig_set(validation_records)
finqa_test_sigs = sig_set(finqa_test_records)
fiqa_test_sigs = sig_set(fiqa_test_records)

leak_checks = {
    "train_vs_validation": len(train_sigs & val_sigs),
    "train_vs_finqa_test": len(train_sigs & finqa_test_sigs),
    "train_vs_fiqa_test": len(train_sigs & fiqa_test_sigs),
}
for name, n in leak_checks.items():
    add_gate(f"Exact input leakage {name}", n == 0, n, 0)

leak_checks


[PASS] Exact input leakage train_vs_validation
[PASS] Exact input leakage train_vs_finqa_test
[PASS] Exact input leakage train_vs_fiqa_test


{'train_vs_validation': 0, 'train_vs_finqa_test': 0, 'train_vs_fiqa_test': 0}

## 16. Optional near-duplicate audit (recommended for final paper run)

This uses MinHash LSH to flag high-overlap train↔evaluation inputs. It is intentionally an **audit**, not an automatic deletion rule: near-duplicates must be inspected because financial templates can be legitimately similar.

In [18]:
def char_ngrams(text: str, n: int = 5) -> set:
    s = normalized_text(text)
    if len(s) < n: return {s}
    return {s[i:i+n] for i in range(len(s)-n+1)}


def run_minhash_audit(train: Sequence[Mapping[str, Any]], eval_records: Sequence[Mapping[str, Any]], threshold=.90):
    try:
        from datasketch import MinHash, MinHashLSH
    except ImportError:
        print("Install datasketch to enable: %pip install datasketch")
        return pd.DataFrame()

    def mh(text):
        m = MinHash(num_perm=128)
        for token in char_ngrams(text):
            m.update(token.encode("utf-8"))
        return m

    lsh = MinHashLSH(threshold=threshold, num_perm=128)
    train_mh = {}
    for i, r in enumerate(tqdm(train, desc="Index train")):
        m = mh(r["input"])
        key = f"tr::{i}"
        lsh.insert(key, m)
        train_mh[key] = r

    hits = []
    for er in tqdm(eval_records, desc="Audit eval"):
        em = mh(er["input"])
        for key in lsh.query(em):
            tr = train_mh[key]
            jac = len(char_ngrams(tr["input"]) & char_ngrams(er["input"])) / max(
                1, len(char_ngrams(tr["input"]) | char_ngrams(er["input"]))
            )
            hits.append({
                "train_id": tr["record_id"], "eval_id": er["record_id"],
                "train_dataset": tr["dataset"], "eval_dataset": er["dataset"],
                "jaccard_char5": jac,
            })
    return pd.DataFrame(hits).sort_values("jaccard_char5", ascending=False) if hits else pd.DataFrame()

near_dup_df = pd.DataFrame()
if RUN_OPTIONAL_NEAR_DUP_AUDIT:
    protected = finqa_test_records + fiqa_test_records
    near_dup_df = run_minhash_audit(train_records, protected, NEAR_DUP_THRESHOLD)
    display(near_dup_df.head(50))

## 17. Export stable Phase-3 / Phase-4 artifacts

The following files are the **only supported downstream data interface**:

- `datasets/train.jsonl`
- `datasets/validation.jsonl`
- `eval/finqa_test.jsonl`
- `eval/fiqa_test.jsonl`
- `eval/fintrust_registry.json`
- `manifests/dataset_registry.json`
- `manifests/phase2_validation_report.json`
- `python/phase2_interface.py`

This minimizes accidental split contamination when Phase-3/4 notebooks evolve.


In [19]:
train_path = EXPORT_DIR / "train.jsonl"
val_path = EXPORT_DIR / "validation.jsonl"
finqa_test_path = EVAL_DIR / "finqa_test.jsonl"
fiqa_test_path = EVAL_DIR / "fiqa_test.jsonl"

write_jsonl(train_records, train_path)
write_jsonl(validation_records, val_path)
write_jsonl(finqa_test_records, finqa_test_path)
write_jsonl(fiqa_test_records, fiqa_test_path)

fintrust_registry = {
    "dataset": "FinTrust",
    "evaluation_only": True,
    "hf_repo": FINTRUST_HF_REPO,
    "github_repo": FINTRUST_GITHUB,
    "dimensions": FINTRUST_DIMENSIONS,
    "data_snapshot": str(fintrust_snapshot.resolve()) if FINTRUST_AVAILABLE else None,
    "official_eval_code": str(fintrust_eval_code.resolve()) if fintrust_eval_code else None,
    "files": fintrust_manifest_df.to_dict(orient="records") if len(fintrust_manifest_df) else [],
    "note": "Use dimension-specific official evaluation/postprocessing; do not convert to SFT training data.",
}
fintrust_registry_path = EVAL_DIR / "fintrust_registry.json"
save_json(fintrust_registry, fintrust_registry_path)

if len(near_dup_df):
    near_dup_df.to_csv(MANIFEST_DIR / "near_duplicate_audit.csv", index=False)

print("Exported:")
for p in [train_path, val_path, finqa_test_path, fiqa_test_path, fintrust_registry_path]:
    print(" -", p)


Exported:
 - /home/heet18/Downloads/Research Papers/artifacts/phase2/datasets/train.jsonl
 - /home/heet18/Downloads/Research Papers/artifacts/phase2/datasets/validation.jsonl
 - /home/heet18/Downloads/Research Papers/artifacts/phase2/eval/finqa_test.jsonl
 - /home/heet18/Downloads/Research Papers/artifacts/phase2/eval/fiqa_test.jsonl
 - /home/heet18/Downloads/Research Papers/artifacts/phase2/eval/fintrust_registry.json


## 18. Generate dataset registry and Phase-3/4 configuration templates

The registry carries counts, hashes, task names and split roles. Phase-3 can use dataset-balanced sampling from this registry; Phase-4 can read protected evaluation paths and keep calibration thresholds restricted to validation data.

In [20]:
def artifact_entry(path: Path, role: str, dataset: str | List[str], task: str | List[str]):
    return {
        "path": str(path.resolve()),
        "role": role,
        "dataset": dataset,
        "task": task,
        "sha256": sha256_file(path),
        "records": sum(1 for _ in path.open("r", encoding="utf-8")) if path.suffix == ".jsonl" else None,
    }

registry = {
    "schema_version": SCHEMA_VERSION,
    "created_unix": int(time.time()),
    "seed": SEED,
    "training_policy": {
        "allowed_datasets": ["FinQA", "FiQA"],
        "forbidden_datasets": ["FinTrust"],
        "recommended_sampling": "dataset-balanced or explicitly reported mixture weights",
    },
    "threshold_policy": {
        "calibration_and_abstention_threshold_selection": "validation only",
        "never_tune_on": ["FinQA test", "FiQA test", "FinTrust"],
    },
    "artifacts": {
        "train": artifact_entry(train_path, "sft_train", ["FinQA", "FiQA"], ["financial_numerical_reasoning", "financial_sentiment_regression"]),
        "validation": artifact_entry(val_path, "validation_and_calibration", ["FinQA", "FiQA"], ["financial_numerical_reasoning", "financial_sentiment_regression"]),
        "finqa_test": artifact_entry(finqa_test_path, "protected_test", "FinQA", "financial_numerical_reasoning"),
        "fiqa_test": artifact_entry(fiqa_test_path, "protected_test", "FiQA", "financial_sentiment_regression"),
        "fintrust_registry": {
            "path": str(fintrust_registry_path.resolve()),
            "role": "protected_trust_evaluation",
            "dataset": "FinTrust",
            "sha256": sha256_file(fintrust_registry_path),
        },
    },
    "source_manifests": {
        "FinQA": finqa_source_manifest,
        "FiQA": fiqa_source_manifest,
        "FinTrust": {
            "hf_repo": FINTRUST_HF_REPO,
            "github_repo": FINTRUST_GITHUB,
            "dimensions": FINTRUST_DIMENSIONS,
        },
    },
}
registry_path = MANIFEST_DIR / "dataset_registry.json"
save_json(registry, registry_path)

phase3_config = {
    "data_registry": str(registry_path.resolve()),
    "train_artifact": "train",
    "validation_artifact": "validation",
    "task_mix": {"FinQA": 0.5, "FiQA": 0.5},
    "mix_note": "Starting point only; report actual sampling strategy in experiments.",
    "target_format": {
        "FinQA": "PROGRAM: <program>\\nANSWER: <answer>",
        "FiQA": "SENTIMENT_SCORE: <number in [-1, 1]>",
    },
}
save_json(phase3_config, MANIFEST_DIR / "phase3_data_config.json")

phase4_config = {
    "data_registry": str(registry_path.resolve()),
    "calibration_split": "validation",
    "protected_tests": ["finqa_test", "fiqa_test", "fintrust_registry"],
    "recommended_metrics": {
        "FinQA": ["answer_accuracy", "program_accuracy", "execution_accuracy", "answer_program_consistency"],
        "FiQA": ["mse", "r2", "mae", "pearson_r"],
        "Calibration": ["ECE", "Brier", "NLL"],
        "SelectivePrediction": ["coverage", "selective_accuracy", "AURC"],
        "FinTrust": ["official_dimension_specific_metrics"],
        "Efficiency": ["trainable_parameters", "peak_vram", "latency", "tokens_per_second"],
    },
    "official_evaluators": {
        "FinQA": "https://github.com/czyssrs/FinQA/tree/main/code/evaluate",
        "FinTrust": FINTRUST_GITHUB,
    },
}
save_json(phase4_config, MANIFEST_DIR / "phase4_eval_config.json")

print("Registry:", registry_path)


Registry: /home/heet18/Downloads/Research Papers/artifacts/phase2/manifests/dataset_registry.json


## 19. Export a tiny Python interface for Phase-3/4

Future notebooks should import this helper rather than copying Phase-2 loading logic.

In [21]:
interface_code = r"""from __future__ import annotations
import json
from pathlib import Path


def _read_jsonl(path):
    path = Path(path)
    out = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                out.append(json.loads(line))
    return out


def load_registry(registry_path):
    with Path(registry_path).open("r", encoding="utf-8") as f:
        return json.load(f)


def load_artifact(registry_path, key):
    reg = load_registry(registry_path)
    entry = reg["artifacts"][key]
    if key == "fintrust_registry":
        with Path(entry["path"]).open("r", encoding="utf-8") as f:
            return json.load(f)
    return _read_jsonl(entry["path"])


def load_train(registry_path):
    return load_artifact(registry_path, "train")


def load_validation(registry_path):
    return load_artifact(registry_path, "validation")


def load_finqa_test(registry_path):
    return load_artifact(registry_path, "finqa_test")


def load_fiqa_test(registry_path):
    return load_artifact(registry_path, "fiqa_test")


def load_fintrust_registry(registry_path):
    return load_artifact(registry_path, "fintrust_registry")
"""
interface_path = MODULE_DIR / "phase2_interface.py"
interface_path.write_text(interface_code, encoding="utf-8")
print("Saved:", interface_path)


Saved: /home/heet18/Downloads/Research Papers/artifacts/phase2/python/phase2_interface.py


# Part E — Final Phase-2 validation
## 20. Artifact-level invariants

These gates re-open exported artifacts, so a successful result verifies the **actual Phase-3/4 inputs**, not only in-memory dataframes.

In [22]:
export_train = read_jsonl(train_path)
export_val = read_jsonl(val_path)
export_finqa_test = read_jsonl(finqa_test_path)
export_fiqa_test = read_jsonl(fiqa_test_path)

add_gate("Exported train has unique record IDs",
         len({r["record_id"] for r in export_train}) == len(export_train))
add_gate("Exported validation has unique record IDs",
         len({r["record_id"] for r in export_val}) == len(export_val))
add_gate("FinQA test marked evaluation-only",
         all(r["metadata"].get("evaluation_only") for r in export_finqa_test))
add_gate("FiQA test marked evaluation-only",
         all(r["metadata"].get("evaluation_only") for r in export_fiqa_test) if export_fiqa_test else FIQA_AVAILABLE)
add_gate("FinTrust registry marked evaluation-only", fintrust_registry.get("evaluation_only") is True)

# Strong protection: no protected test source IDs occur in training.
train_ids = {r["record_id"] for r in export_train}
protected_ids = {r["record_id"] for r in export_finqa_test + export_fiqa_test}
add_gate("No protected test record IDs in training", len(train_ids & protected_ids) == 0,
         len(train_ids & protected_ids), 0)


[PASS] Exported train has unique record IDs
[PASS] Exported validation has unique record IDs
[PASS] FinQA test marked evaluation-only
[PASS] FiQA test marked evaluation-only
[PASS] FinTrust registry marked evaluation-only
[PASS] No protected test record IDs in training


Gate(name='No protected test record IDs in training', passed=True, observed=0, expected=0, severity='ERROR', note='')

## 21. Phase-2 PASS/FAIL report

A research run is considered **PASS** only when every `ERROR` gate passes. `WARNING` gates are retained in the report but do not block export. If the FiQA source has not been configured, this notebook intentionally ends with `PHASE-2 STATUS: FAIL` — the artifacts remain useful for debugging, but Phase-3 training should not be started from an incomplete benchmark setup.


In [23]:
report_rows = [asdict(g) for g in GATES]
report_df = pd.DataFrame(report_rows)
display(report_df)

hard_failures = [g for g in GATES if g.severity == "ERROR" and not g.passed]
phase2_pass = len(hard_failures) == 0

validation_report = {
    "schema_version": SCHEMA_VERSION,
    "phase2_pass": phase2_pass,
    "hard_failures": [asdict(g) for g in hard_failures],
    "gates": report_rows,
    "artifact_registry": str(registry_path.resolve()),
    "interface_module": str(interface_path.resolve()),
}
validation_report_path = MANIFEST_DIR / "phase2_validation_report.json"
save_json(validation_report, validation_report_path)

print("=" * 80)
print("PHASE-2 STATUS:", "PASS" if phase2_pass else "FAIL")
print("Hard failures:", len(hard_failures))
for g in hard_failures:
    print(" -", g.name, "| observed:", g.observed, "| expected:", g.expected)
print("Validation report:", validation_report_path)
print("=" * 80)

,name,passed,observed,expected,severity,note
0,FinQA train count,True,6251,6251,ERROR,
1,FinQA dev count,True,883,883,ERROR,
2,FinQA test count,True,1147,1147,ERROR,
3,FinQA IDs globally unique,True,8281,8281,ERROR,
4,FinQA required field present: id,True,0,0,ERROR,
5,FinQA required field present: question,True,0,0,ERROR,
6,FinQA required field present: exe_ans,True,0,0,ERROR,
7,FinQA required field present: program,True,0,0,ERROR,
8,FinQA required field present: gold_inds,True,0,0,ERROR,
9,FinQA required field present: table,True,0,0,ERROR,


PHASE-2 STATUS: PASS
Hard failures: 0
Validation report: /home/heet18/Downloads/Research Papers/artifacts/phase2/manifests/phase2_validation_report.json


## 22. Phase-3 / Phase-4 handoff contract

### Phase-3 — Baselines and QLoRA
Use only:
- `datasets/train.jsonl`
- `datasets/validation.jsonl`
- `manifests/phase3_data_config.json`

Recommended first experiments:
1. base SLM zero-shot,
2. few-shot,
3. FinQA-only QLoRA,
4. FiQA-only QLoRA,
5. joint FinQA+FiQA QLoRA with reported sampling weights.

### Phase-4 — TrustFin-SLM
Use:
- validation JSONL for temperature scaling / confidence threshold selection,
- protected FinQA test for answer/program/execution consistency,
- protected FiQA test for MAE/Pearson/calibration,
- FinTrust registry + official dimension-specific evaluator for trust assessment.

### What this Phase-2 deliberately does **not** do
- It does not fabricate benchmark data or substitute FOMC for FiQA.
- It does not train on FinTrust.
- It does not place FinQA gold evidence into test prompts.
- It does not generate natural-language chain-of-thought targets.
- It does not tune abstention/calibration thresholds on test data.

When `PHASE-2 STATUS: PASS`, Phase-3/4 can be developed independently while preserving the same frozen data contract.
